In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Define the industries of GICS classification
gics_industries = {
    "101010": "Energy Equipment & Services",
    "101020": "Oil, Gas & Consumable Fuels",
    "151010": "Chemicals",
    "151020": "Construction Materials",
    "151030": "Containers & Packaging",
    "151040": "Metals & Mining",
    "151050": "Paper & Forest Products",
    "201010": "Aerospace & Defense",
    "201020": "Building Products",
    "201030": "Construction & Engineering",
    "201040": "Electrical Equipment",
    "201050": "Industrial Conglomerates",
    "201060": "Machinery",
    "201070": "Trading Companies & Distributors",
    "202010": "Commercial Services & Supplies",
    "202020": "Professional Services",
    "203010": "Air Freight & Logistics",
    "203020": "Passenger Airlines",
    "203030": "Marine Transportation",
    "203040": "Ground Transportation",
    "203050": "Transportation Infrastructure",
    "251010": "Automobile Components",
    "251020": "Automobiles",
    "252010": "Household Durables",
    "252020": "Leisure Products",
    "252030": "Textiles, Apparel & Luxury Goods",
    "253010": "Hotels, Restaurants & Leisure",
    "253020": "Diversified Consumer Services",
    "255010": "Distributors",
    "255030": "Broadline Retail",
    "255040": "Specialty Retail",
    "301010": "Consumer Staples Distribution & Retail",
    "302010": "Beverages",
    "302020": "Food Products",
    "302030": "Tobacco",
    "303010": "Household Products",
    "303020": "Personal Care Products",
    "351010": "Health Care Equipment & Supplies",
    "351020": "Health Care Providers & Services",
    "351030": "Health Care Technology",
    "352010": "Biotechnology",
    "352020": "Pharmaceuticals",
    "352030": "Life Sciences Tools & Services",
    "401010": "Banks",
    "402010": "Financial Services",
    "402020": "Consumer Finance",
    "402030": "Capital Markets",
    "402040": "Mortgage Real Estate Investment Trusts (REITs)",
    "403010": "Insurance",
    "451020": "IT Services",
    "451030": "Software",
    "452010": "Communications Equipment",
    "452020": "Technology Hardware, Storage & Peripherals",
    "452030": "Electronic Equipment, Instruments & Components",
    "453010": "Semiconductors & Semiconductor Equipment",
    "501010": "Diversified Telecommunication Services",
    "501020": "Wireless Telecommunication Services",
    "502010": "Media",
    "502020": "Entertainment",
    "502030": "Interactive Media & Services",
    "551010": "Electric Utilities",
    "551020": "Gas Utilities",
    "551030": "Multi-Utilities",
    "551040": "Water Utilities",
    "551050": "Independent Power and Renewable Electricity Producers",
    "601010": "Diversified REITs",
    "601025": "Industrial REITs",
    "601030": "Hotel & Resort REITs",
    "601040": "Office REITs",
    "601050": "Health Care REITs",
    "601060": "Residential REITs",
    "601070": "Retail REITs",
    "601080": "Specialized REITs",
    "602010": "Real Estate Management & Development",
    "na" : "Unclassified"
}

In [ ]:
# Duplicate stocks in the S&P 500
# duplicate_stocks = { 
#     "50203010" : "GOOGL",
#     "50202010" : "DISCK",
#     "50201020" : "FOXA",
#     "50201040" : "NWSA"
#     }

duplicate_stocks = {}

In [ ]:
# Get the industry of a company based on its GICS code
def get_industry(gics_code):
    
    if gics_code == np.nan:
        return "na"
    
    gics_code = str(gics_code)
    if gics_code[:6] in gics_industries:
        return gics_code[:6]
    else:
        return "na"

In [ ]:
# Replace column names with actual gics codes
tickers = pd.read_csv('../Data/data_us/tickers.csv', header=None, dtype=str)
tickers.columns = ['ticker', 'gics_code']
tickers.set_index('ticker', inplace=True)

tickers['industry'] = tickers['gics_code'].apply(get_industry)

display(tickers)

In [ ]:
close_prices = pd.read_csv('../Data/data_us/adjusted.csv')

close_prices['Date'] = pd.to_datetime(close_prices['Date'], format = "%Y%m%d")
close_prices = close_prices.set_index('Date')

close_prices.columns = close_prices.columns.str.strip()

# for key, value in duplicate_stocks.items():
#     close_prices.drop(value, axis=1, inplace=True)

display(close_prices.head())

In [ ]:
univ_list = pd.read_csv("../Data/data_us/univ_h.csv")
univ_list['year'] = pd.to_datetime(univ_list['year'], format = "%Y")
univ_list.set_index('year', inplace= True)

close_prices_dict = {}
stocks_list_dict = {}
volatility_dict = {}


for year in univ_list.index.unique():
    stocks_list = univ_list.columns[univ_list.loc[year] == 1].tolist()
    
    stocks_list = [stock for stock in stocks_list if stock in close_prices.columns]
    print(f"Year: {year.year} - Number of stocks: {len(stocks_list)}")
    
    stocks_list_dict[year.year] = stocks_list
    close_prices_dict[year.year] = close_prices
    
    temp = close_prices_dict[year.year].loc[str(year.year)]
    
    # Add last month from previous year if present
    if year.year != 2004:
        previous_year_end = close_prices.loc[str(year.year - 1)].tail(30)
        temp = pd.concat([previous_year_end, close_prices_dict[year.year]])
    
    # Add first 10 days from next year if present
    if year.year != 2024:
        next_year_start = close_prices.loc[str(year.year + 1)].head(10)
        temp = pd.concat([close_prices_dict[year.year], next_year_start])
        
    close_prices_dict[year.year] = temp


for year in univ_list.index.unique():
    stocks_list = stocks_list_dict[year.year]
    close_prices_dict[year.year] = close_prices_dict[year.year][stocks_list]
    
    
display(close_prices_dict[2005].head(10))
display(close_prices_dict[2005].tail(10))
    


In [ ]:
volatility_dict = {}
five_day_return_dict = {}
norm_five_day_return_dict = {}
industry_v_dict = {}
returns_minus_industry_dict = {}
industry_returns_dict = {}
beta_dict = {}
ranked_beta_dict = {}
rsquared_dict = {}
ranked_rsquared_dict = {}
ranked_norm_five_day_return_dict = {}
industry_ranked_v_dict = {}

for year in close_prices_dict.keys():
    if year == 2004:
        continue
    else:
        volatility_dict[year] = (np.log(close_prices_dict[year]).diff(1).fillna(0)).rolling(window=21).std()
        volatility_dict[year] = volatility_dict[year].applymap(lambda x: max(0.005, x))
        five_day_return_dict[year] = (np.log(close_prices_dict[year]).diff(5).fillna(0))
        norm_five_day_return_dict[year] = five_day_return_dict[year] / volatility_dict[year]
        
        returns_minus_industry_dict[year] = (np.log(close_prices_dict[year]).diff(1).fillna(0))
        
        ranks = norm_five_day_return_dict[year].rank(axis=1, method = 'first', ascending=False)
        N = ranks.shape[1]
        
        # Redefine the factor in terms of the rank
        ranked_norm_five_day_return_dict[year] = (N + 1 - 2 * ranks) / (N - 1)
        # display(ranked_norm_five_day_return_dict[year].head())

        # norm_five_day_return_dict[year] = norm_five_day_return_dict[year].loc[str(year)]
        # returns_minus_industry_dict[year] = returns_minus_industry_dict[year].loc[str(year)]
        # ranked_norm_five_day_return_dict[year] = ranked_norm_five_day_return_dict[year].loc[str(year)]
        
        industry_v_dict[year] = {}
        industry_returns_dict[year] = {}
        industry_ranked_v_dict[year] = {}
        
        for key, value in gics_industries.items():
            mask = tickers[tickers['industry'] == key].index
            mask = [ticker for ticker in mask if ticker in norm_five_day_return_dict[year].columns]

            if mask:
                
                industry_v_dict[year][key] = norm_five_day_return_dict[year].loc[:, mask].mean(axis=1)
                industry_returns_dict[year][key] = returns_minus_industry_dict[year].loc[:, mask].mean(axis=1)
                industry_ranked_v_dict[year][key] = ranked_norm_five_day_return_dict[year].loc[:, mask].mean(axis=1)

                # subtract the industry average from the stock return
                norm_five_day_return_dict[year].loc[:, mask] = norm_five_day_return_dict[year].loc[:, mask].sub(industry_v_dict[year][key], axis=0)
                
                ranked_norm_five_day_return_dict[year].loc[:, mask] = ranked_norm_five_day_return_dict[year].loc[:, mask].sub(industry_ranked_v_dict[year][key], axis=0)

                # Subtract the industry returns
                returns_minus_industry_dict[year].loc[:, mask] = returns_minus_industry_dict[year].loc[:, mask].sub(industry_returns_dict[year][key], axis=0)

        norm_five_day_return_dict[year] = norm_five_day_return_dict[year].loc[str(year)]
        ranked_norm_five_day_return_dict[year] = ranked_norm_five_day_return_dict[year].loc[str(year)]
        returns_minus_industry_dict[year] = returns_minus_industry_dict[year].shift(-1).loc[str(year)]
        
            
        beta_dict[year] = {}
        rsquared_dict[year] = {}
        ranked_beta_dict[year] = {}
        ranked_rsquared_dict[year] = {}
        
        for t in norm_five_day_return_dict[year].index:
            # Dependent variable: returns at time t+1
            R_t1 = returns_minus_industry_dict[year].loc[t]
            
            # Independent variable: factors at time t
            v_t = norm_five_day_return_dict[year].loc[t]
            
            ranked_v_t = ranked_norm_five_day_return_dict[year].loc[t]
            
             # Calculate beta(t)
            beta_t = (R_t1 * v_t).sum() / (v_t * v_t).sum()
            beta_dict[year][t] = beta_t
            
            ranked_beta_t = (R_t1 * ranked_v_t).sum() / (ranked_v_t * ranked_v_t).sum()
            ranked_beta_dict[year][t] = ranked_beta_t
            
            
            # Calculate residuals
            epsilon_t = R_t1 - beta_t * v_t
            ranked_epsilon_t = R_t1 - ranked_beta_t * ranked_v_t
            
            # Calculate R^2(t)
            rsquared_t = 1 - (epsilon_t ** 2).sum() / (R_t1 * R_t1).sum()
            rsquared_dict[year][t] = rsquared_t
            
            ranked_rsquared_t = 1 - (ranked_epsilon_t ** 2).sum() / (R_t1 * R_t1).sum()
            ranked_rsquared_dict[year][t] = ranked_rsquared_t
            
            
        # Convert the dictionaries to dataframes
        beta_dict[year] = pd.Series(beta_dict[year])
        rsquared_dict[year] = pd.Series(rsquared_dict[year])
        ranked_beta_dict[year] = pd.Series(ranked_beta_dict[year])
        ranked_rsquared_dict[year] = pd.Series(ranked_rsquared_dict[year])
        
# drop last row for 2024 from beta list
beta_dict[2024] = beta_dict[2024].iloc[:-1]
rsquared_dict[2024] = rsquared_dict[2024].iloc[:-1]
ranked_beta_dict[2024] = ranked_beta_dict[2024].iloc[:-1]
ranked_rsquared_dict[2024] = ranked_rsquared_dict[2024].iloc[:-1]

print("Beta")
display(beta_dict[2005].head())
display(beta_dict[2005].tail())

display(beta_dict[2006].tail())

display(beta_dict[2024].tail())

print("R-squared")
display(rsquared_dict[2005].head())
display(rsquared_dict[2024].tail())

returns_minus_industry_dict[2005].tail()
returns_minus_industry_dict[2024].tail()
        


In [ ]:
summary_df = pd.DataFrame(columns=['Year', 'avg_beta', 't_stat', 'T'])

for year in beta_dict.keys():
    avg_beta = beta_dict[year].mean()
    T = len(beta_dict[year])
    t_stat = np.sqrt(T) * avg_beta / beta_dict[year].std()
    
    
    summary_df.loc[len(summary_df)] = [str(year), avg_beta, t_stat, T]

summary_df.set_index('Year', inplace=True)
display(summary_df)

In [ ]:
ranked_summary_df = pd.DataFrame(columns=['Year', 'avg_beta', 't_stat', 'T'])

for year in ranked_beta_dict.keys():
    avg_beta = ranked_beta_dict[year].mean()
    T = len(ranked_beta_dict[year])
    t_stat = np.sqrt(T) * avg_beta / ranked_beta_dict[year].std()
    
    
    ranked_summary_df.loc[len(ranked_summary_df)] = [str(year), avg_beta, t_stat, T]

ranked_summary_df.set_index('Year', inplace=True)
display(ranked_summary_df)

In [ ]:
expected_returns_dict = {}
portfolio_returns_dict = {}

for year in range(2006, 2025):
    previous_year = str(year - 1)
    expected_returns_dict[year] = summary_df.loc[previous_year, 'avg_beta'] * norm_five_day_return_dict[year]
    portfolio_returns_dict[year] = {}
    
    for t in expected_returns_dict[year].index:
        
        # Find top 20% of stocks
        top_20 = expected_returns_dict[year].loc[t].nlargest(int(len(expected_returns_dict[year].loc[t]) * 0.2)).index.tolist()
        
        # Find bottom 20% of stocks
        bottom_20 = expected_returns_dict[year].loc[t].nsmallest(int(len(expected_returns_dict[year].loc[t]) * 0.2)).index.tolist()
        
        portfolio_returns_dict[year][t] = (np.log(close_prices_dict[year]).diff(1)[[stock for stock in top_20]].loc[t].sum() - np.log(close_prices_dict[year]).diff(1)[[stock for stock in bottom_20]].loc[t].sum())/len(top_20)
        
    portfolio_returns_dict[year] = pd.DataFrame.from_dict(portfolio_returns_dict[year], orient='index', columns=['portfolio_returns'])
    portfolio_returns_dict[year].index = pd.to_datetime(portfolio_returns_dict[year].index)
    portfolio_returns_dict[year] = portfolio_returns_dict[year].loc[str(year)]    
    
display(portfolio_returns_dict[2006].head())
display(portfolio_returns_dict[2024].tail())

In [ ]:
plt.figure(figsize=(15, 10))
plt.title("Portfolio Returns")
plt.xlabel("Date")
plt.ylabel("Returns")
plt.legend(title="Year")

for year in portfolio_returns_dict.keys():
    returns = (1 + portfolio_returns_dict[year]).cumprod() - 1
    # Plot number of days in a year vs returns
    plt.plot(range(1, len(returns)+1), returns.values, label=year)
    if not returns.empty:
        print(f"Year: {year} - Portfolio Returns: {returns.iloc[-1].values[0]}")

plt.legend(title="Year")
plt.show()